# 03 — Feature Engineering

This notebook builds model-ready features and produces the final train/test CSVs:
1. Load the cleaned checkpoint
2. Date-based features (year, month, day, day-of-week, season)
3. Lag & rolling-average features
4. Encode categorical columns (City, Season, AQI_Bucket)
5. Scale numeric features
6. Chronological train/test split
7. Save to `data/processed/`

## 1. Import


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib


## 2. Load the clean data


In [3]:
df = pd.read_csv("../data/interim/city_day_nulls_handled.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['City', 'Date']).reset_index(drop=True)
df.shape


(29531, 16)

## 3. Data based feature


In [7]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6,7,8,9]:
        return "Monsoon"
    else:
        return "Post Monsoon"

df['Season'] = df['Month'].apply(get_season)
df[['Date','Year','Month','Day','DayOfWeek','Season']].tail()

,Date,Year,Month,Day,DayOfWeek,Season
29526,2020-06-27,2020,6,27,5,Monsoon
29527,2020-06-28,2020,6,28,6,Monsoon
29528,2020-06-29,2020,6,29,0,Monsoon
29529,2020-06-30,2020,6,30,1,Monsoon
29530,2020-07-01,2020,7,1,2,Monsoon


## 4. Lag & rolling-average features
These give the model recent-history context, which matters a lot for AQI since pollution levels are strongly autocorrelated day to day.

In [8]:
df['AQI_lag1'] = df.groupby('City')['AQI'].shift(1)
df['AQI_rolling7'] = df.groupby('City')['AQI'].transform(lambda x: x.rolling(7, min_periods=1).mean())

# the first row per city has no previous day, so AQI_lag1 will be NaN there — drop those rows
before = len(df)
df = df.dropna(subset=['AQI_lag1'])
print(f"Dropped {before - len(df)} rows with no lag value (first day per city)")


Dropped 4705 rows with no lag value (first day per city)
